# Experiment 4 — RQ3: Universal vs. Generator-Specific Forensic Features

**Research Question:**  
Are there handcrafted forensic features that remain consistently informative across different AI image generators, or are the most important features generator-dependent?

**What this notebook does:**
1. Clones the project repo from GitHub and mounts Google Drive
2. Installs required dependencies
3. Verifies feature data is accessible
4. Runs the full RQ3 pipeline via `run_exp4()` — a single call that executes all 4 phases:
   - **Phase 1:** Metadata loading → Generator-wise Mutual Information → Full feature statistics
   - **Phase 2:** Universal feature selection, Generator-specific feature detection, Family ranking
   - **Phase 3:** LightGBM family ablation + Spearman rank correlation validation
   - **Phase 4:** All RQ3 visualizations (4 plots)
5. Displays all key results and saved output files inline

**Output files saved to Drive:**
```
outputs/exp4_rq3/
  ├── mi_scores_per_generator.csv
  ├── feature_stability.csv
  ├── universal_features.csv
  ├── generator_specific_features.csv
  ├── feature_family_ranking.csv
  ├── feature_family_ablation.csv
  └── figures/
      ├── mi_heatmap_families.png
      ├── family_avg_mi_bar.png
      ├── stability_scatter.png
      └── ablation_f1_bar.png
```

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted.')

In [ ]:
# ── Cell 2: Clone repo from GitHub & add to Python path ──────────────────────
import sys
import os
from pathlib import Path

REPO_URL  = 'https://github.com/MaishaNajAlam/cross-generator-ai-image-detection.git'
REPO_DIR  = Path('/content/cross-generator-ai-image-detection')

if not REPO_DIR.exists():
    print('Cloning repository...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print('Repository already cloned — pulling latest changes...')
    os.system(f'git -C {REPO_DIR} pull')

# Add src/ to path so all module imports work
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    sys.path.insert(0, str(REPO_DIR))

print(f'✓ Repo ready at: {REPO_DIR}')
print(f'✓ src/ added to sys.path')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
!pip install -q lightgbm scikit-learn seaborn matplotlib pandas numpy scipy
print('✓ Dependencies installed.')

In [ ]:
# ── Cell 4: Configure paths (edit FEATURES_ROOT and OUTPUT_DIR if needed) ─────
from pathlib import Path

# Root directory where your extracted feature .npy files are stored on Drive
# Must contain: train/, test/, val/, metadata/feature_metadata.json, metadata/feature_names.txt
FEATURES_ROOT = '/content/drive/MyDrive/ml_project/processed/features/v1'

# Where all RQ3 outputs (CSVs + figures) will be saved
OUTPUT_DIR = '/content/drive/MyDrive/ml_project/outputs/exp4_rq3'

print(f'FEATURES_ROOT : {FEATURES_ROOT}')
print(f'OUTPUT_DIR    : {OUTPUT_DIR}')

# Quick check that the features directory exists
if not Path(FEATURES_ROOT).exists():
    print('\n⚠️  WARNING: FEATURES_ROOT does not exist. Check your Drive path.')
else:
    splits = [d.name for d in Path(FEATURES_ROOT).iterdir() if d.is_dir()]
    print(f'\n✓ Features directory found. Subdirectories: {splits}')

In [ ]:
# ── Cell 5: Verify feature data & metadata files ──────────────────────────────
import numpy as np
from pathlib import Path

features_root = Path(FEATURES_ROOT)
required_files = [
    features_root / 'train' / 'combined.npy',
    features_root / 'train' / 'labels.npy',
    features_root / 'train' / 'generators.npy',
    features_root / 'test'  / 'combined.npy',
    features_root / 'test'  / 'labels.npy',
    features_root / 'test'  / 'generators.npy',
    features_root / 'metadata' / 'feature_metadata.json',
    features_root / 'metadata' / 'feature_names.txt',
]

all_ok = True
for f in required_files:
    status = '✓' if f.exists() else '✗  MISSING'
    if not f.exists():
        all_ok = False
    print(f'  {status}  {f.relative_to(features_root)}')

if all_ok:
    X_train = np.load(features_root / 'train' / 'combined.npy', mmap_mode='r')
    y_train = np.load(features_root / 'train' / 'labels.npy')
    g_train = np.load(features_root / 'train' / 'generators.npy')
    print(f'\n✓ Train split loaded:')
    print(f'   X_train shape    : {X_train.shape}  (N samples × D features)')
    print(f'   y_train unique   : {np.unique(y_train).tolist()}  (0=Real, 1=Fake)')
    print(f'   Generators found : {np.unique(g_train).tolist()}')
else:
    print('\n⚠️  Some required files are missing. Resolve before running Cell 6.')

In [ ]:
# ── Cell 6: RUN EXPERIMENT 4 (RQ3) ───────────────────────────────────────────
# This single call executes all 4 phases end-to-end.
# Runtime estimate: ~15–40 minutes depending on dataset size and Colab resources.
#   Phase 1 (MI computation)  — slowest: 5× mutual_info_classif over full D features
#   Phase 3 (LightGBM ablation) — 7 families × 20 off-diagonal train/test pairs

from experiments.exp4_rq3 import run_exp4

results = run_exp4(
    features_root=FEATURES_ROOT,
    output_dir=OUTPUT_DIR,
    info_threshold=0.05,               # Min mean_mi to be called 'universal'
    specificity_ratio_threshold=2.0,   # Min max_mi/mean_mi to be called 'specific'
    max_mi_threshold=0.10,             # Min max_mi for generator-specific features
    top_n=20,                          # Top N features in each output table
)

print('\n✓ Experiment 4 complete!')
print(f"  Spearman r (MI vs F1 ranking) = {results['spearman_r']:.4f}")
print(f"  All outputs saved to: {results['output_dir']}")

In [ ]:
# ── Cell 7: Display Feature Family Ranking ────────────────────────────────────
import pandas as pd

print('Feature Family Ranking (Average MI per feature — normalized):')
print('=' * 70)
display(results['family_pivot_df'].style.format('{:.4f}', subset=pd.IndexSlice[:, results['family_pivot_df'].columns[:-1]]).background_gradient(cmap='YlOrRd', axis=None))

In [ ]:
# ── Cell 8: Display Top Universal Features ────────────────────────────────────
print(f"Top Universal Features (mean_mi > 0.05, ranked by stability_score):")
print('=' * 70)
display(
    results['universal_df'][['feature_name', 'family', 'mean_mi', 'std_mi', 'stability_score']]
    .style.format({'mean_mi': '{:.4f}', 'std_mi': '{:.4f}', 'stability_score': '{:.2f}'})
    .background_gradient(subset=['stability_score'], cmap='Greens')
)
print(f"  Family distribution: {results['universal_df']['family'].value_counts().to_dict()}")

In [ ]:
# ── Cell 9: Display Top Generator-Specific Features ───────────────────────────
print(f"Top Generator-Specific Features (max_mi/mean_mi >= 2.0, ranked by max_mi):")
print('=' * 70)
display(
    results['specific_df'][['feature_name', 'family', 'max_mi', 'mean_mi', 'specificity_ratio']]
    .style.format({'max_mi': '{:.4f}', 'mean_mi': '{:.4f}', 'specificity_ratio': '{:.2f}'})
    .background_gradient(subset=['specificity_ratio'], cmap='Oranges')
)
print(f"  Family distribution: {results['specific_df']['family'].value_counts().to_dict()}")

In [ ]:
# ── Cell 10: Display Family Ablation Results + Spearman Correlation ───────────
print('Family-wise LightGBM Ablation (off-diagonal cross-generator F1):')
print('=' * 70)
display(
    results['ablation_df']
    .style.format({'avg_cross_gen_f1': '{:.4f}'})
    .background_gradient(subset=['avg_cross_gen_f1'], cmap='Blues')
)

r = results['spearman_r']
if r > 0.6:
    interp = 'STRONG — MI conclusions are validated by ablation.'
elif r > 0.4:
    interp = 'MODERATE — MI is a reasonable proxy for F1 ranking.'
else:
    interp = 'WEAK — MI-based conclusions should be treated cautiously.'

print(f'\nSpearman Rank Correlation (MI ranking vs F1 ranking):')
print(f'  r = {r:.4f}')
print(f'  → {interp}')

In [ ]:
# ── Cell 11: Display all saved figures inline ─────────────────────────────────
from pathlib import Path
from IPython.display import Image, display

fig_dir = Path(OUTPUT_DIR) / 'figures'
figures = [
    ('mi_heatmap_families.png',  'Plot 1: Average MI per Feature — Family × Generator'),
    ('family_avg_mi_bar.png',    'Plot 2: Feature Family Ranking (avg MI, cross-generator)'),
    ('stability_scatter.png',    'Plot 3: Feature Stability Scatter (mean_mi vs std_mi)'),
    ('ablation_f1_bar.png',      'Plot 4: Family-wise Ablation — Cross-Generator F1 (LightGBM)'),
]

for fname, title in figures:
    fpath = fig_dir / fname
    if fpath.exists():
        print(f'\n{title}')
        print('-' * len(title))
        display(Image(filename=str(fpath), width=750))
    else:
        print(f'⚠️  {fname} not found — check Phase 4 output.')

In [ ]:
# ── Cell 12: Verify all output files were saved ───────────────────────────────
from pathlib import Path

output_root = Path(OUTPUT_DIR)
expected = [
    'mi_scores_per_generator.csv',
    'feature_stability.csv',
    'universal_features.csv',
    'generator_specific_features.csv',
    'feature_family_ranking.csv',
    'feature_family_ablation.csv',
    'figures/mi_heatmap_families.png',
    'figures/family_avg_mi_bar.png',
    'figures/stability_scatter.png',
    'figures/ablation_f1_bar.png',
]

print('Output file verification:')
all_present = True
for rel_path in expected:
    fpath = output_root / rel_path
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f'  ✓  {rel_path:<45} ({size_kb:.1f} KB)')
    else:
        print(f'  ✗  MISSING: {rel_path}')
        all_present = False

print()
if all_present:
    print('✓ All expected outputs are present.')
else:
    print('⚠️  Some outputs are missing — check the logs above for errors.')